### **Q1. What is Gradient Boosting Regression?**  

Gradient Boosting Regression is a machine learning technique that builds an ensemble of weak learners (typically decision trees) to improve predictive accuracy. Unlike AdaBoost, which assigns weights to misclassified samples, Gradient Boosting optimizes a loss function using gradient descent. It works by sequentially training new models to predict the residual errors of the previous models, gradually reducing the overall error.  

### **Q2. Implement a simple gradient boosting algorithm from scratch using Python and NumPy.**  

Below is an implementation of Gradient Boosting Regression from scratch using Python and NumPy. We use a simple dataset and evaluate the model using Mean Squared Error (MSE) and R-squared (\( R^2 \)).  

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Generate synthetic dataset
np.random.seed(42)
X = np.linspace(0, 10, 100).reshape(-1, 1)
y = np.sin(X).ravel() + np.random.normal(0, 0.1, X.shape[0])  # Adding noise

class GradientBoostingRegressor:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.models = []

    def fit(self, X, y):
        # Initialize the model with the mean of the target values
        self.f0 = np.mean(y)
        residuals = y - self.f0  # Initial residuals

        for _ in range(self.n_estimators):
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.models.append(tree)
            predictions = tree.predict(X)
            residuals -= self.learning_rate * predictions  # Update residuals

    def predict(self, X):
        predictions = np.full(X.shape[0], self.f0)
        for tree in self.models:
            predictions += self.learning_rate * tree.predict(X)
        return predictions

# Train and evaluate the model
gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3)
gbr.fit(X, y)
y_pred = gbr.predict(X)

# Evaluate performance
mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R-squared: {r2:.4f}")

# Plot results
plt.scatter(X, y, label="True values", color='blue', alpha=0.5)
plt.plot(X, y_pred, label="GBR Predictions", color='red')
plt.legend()
plt.show()
```

### **Q3. Experiment with different hyperparameters**  

To optimize the model, we can use **Grid Search** or **Random Search** to find the best hyperparameters such as:  
- **Number of estimators** (\(n_{\text{estimators}}\))  
- **Learning rate** (\(\eta\))  
- **Max depth of trees** (\(d\))  

#### **Grid Search Implementation**  
```python
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor

param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [2, 3, 5]
}

gb = GradientBoostingRegressor()
grid_search = GridSearchCV(gb, param_grid, scoring='r2', cv=5)
grid_search.fit(X, y)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best R-squared Score: {grid_search.best_score_:.4f}")
```

#### **Random Search Implementation**
```python
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(gb, param_grid, scoring='r2', n_iter=5, cv=5, random_state=42)
random_search.fit(X, y)

print(f"Best Parameters (Random Search): {random_search.best_params_}")
print(f"Best R-squared Score: {random_search.best_score_:.4f}")
```

### **Q4. What is a weak learner in Gradient Boosting?**  

A weak learner in Gradient Boosting is a simple model that performs slightly better than random guessing. Typically, weak learners are decision trees with a limited depth (e.g., **stumps**, which are trees with only one split). These weak learners are sequentially improved by focusing on reducing errors in previous iterations.

### **Q5. What is the intuition behind the Gradient Boosting algorithm?**  

The key idea behind Gradient Boosting is that instead of fitting a model to the target variable directly, we fit a sequence of models to the **residual errors**. This is based on the principle that each new model should correct the mistakes made by the previous ones.  

**Steps of intuition:**  
1. **Start with an initial prediction** – Often the mean of the target variable.  
2. **Compute residuals** – Find the difference between actual and predicted values.  
3. **Fit a weak model to residuals** – This model predicts how much correction is needed.  
4. **Update predictions** – Add the weak model’s prediction (scaled by a learning rate) to the previous model.  
5. **Repeat steps 2-4** – The model continuously improves by focusing on reducing errors in a gradient descent fashion.  

### **Q6. How does Gradient Boosting algorithm build an ensemble of weak learners?**  

Gradient Boosting builds an ensemble by sequentially adding weak learners in the following manner:  

1. **Initialize the model** – Start with a simple prediction (e.g., mean of target values).  
2. **Compute the residuals** – Find errors from the previous model.  
3. **Train a weak learner** – Fit a weak model (e.g., decision tree) to predict the residuals.  
4. **Update the model** – Add the weighted contribution of the weak learner to the final prediction.  
5. **Repeat the process** – Continue training new weak learners until convergence or reaching a predefined number of iterations.  

Each weak model is trained to minimize the loss function by using gradient descent, ensuring that subsequent learners improve on previous errors.  

### **Q7. What are the steps involved in constructing the mathematical intuition of Gradient Boosting algorithm?**  

Gradient Boosting is mathematically framed as an optimization problem where we minimize a loss function \( L(y, F(x)) \) using gradient descent. The steps involved are:  

1. **Initialize the model with a constant prediction**  
   $$
   F_0(x) = \arg\min_c \sum_{i=1}^{n} L(y_i, c)
   $$  
   Typically, this is the mean of the target variable for regression problems.  

2. **Compute residuals (negative gradients of the loss function)**  
   $$
   r_i^m = - \frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}
   $$  
   These residuals represent the direction in which the model should improve.  

3. **Train a weak learner (decision tree) on residuals**  
   $$
   h_m(x) = \arg\min_h \sum_{i=1}^{n} (r_i^m - h(x_i))^2
   $$  

4. **Update the model using learning rate \( \eta \)**  
   $$
   F_m(x) = F_{m-1}(x) + \eta h_m(x)
   $$
   Here, \( \eta \) (learning rate) controls how much each weak learner contributes to the final model.  

5. **Repeat steps 2-4** until convergence or reaching the maximum number of estimators.  

This iterative approach ensures that each new weak learner refines the prediction by following the steepest gradient direction of the loss function.  
